In [1]:
# ========== 导入：合成数据生成所需的核心库 ==========
# PyTorch：张量运算、CUDA 设备、dtype（如 float16）
import torch
# Gradio：后面用 Interface 做简易 Web UI（本格先导入备用）
import gradio as gr
# AutoTokenizer / AutoModelForCausalLM：按 model id 自动选类加载
# BitsAndBytesConfig：4bit 量化配置，降低显存占用
from transformers import AutoTokenizer, AutoModelForCausalLM,BitsAndBytesConfig


In [2]:
# ========== 安装 accelerate：大模型加载/设备放置常用依赖 ==========
# -U：升级；-q：安静；版本下界 1.1.0 与 transformers 生态配套
!pip install -U -q "accelerate>=1.1.0"


In [3]:
# ========== 安装 bitsandbytes：4bit / 8bit 量化后端 ==========
# 没有它，BitsAndBytesConfig(load_in_4bit=True) 通常无法工作
!pip install -U -q "bitsandbytes>=0.46.1"


In [2]:
# ========== 4bit 量化配置：面向约 12GB 显存（VRAM）场景 ==========
# BitsAndBytesConfig：告诉 from_pretrained 如何做量化加载
quant_config = BitsAndBytesConfig(
    # 启用 4bit 权重加载，显著省显存
    load_in_4bit=True,
    # nf4：NormalFloat4，常见高质量 4bit 量化类型
    bnb_4bit_quant_type="nf4",
    # 计算时用 float16，在精度与速度间折中
    bnb_4bit_compute_dtype=torch.float16,
    # 双重量化：再压缩量化常数，进一步省显存
    bnb_4bit_use_double_quant=True
)


In [3]:
# ========== 加载 tokenizer + 因果语言模型到 GPU ==========
# Hugging Face Hub 上的模型 id（字符串必须原样）
model_name = "ehartford/WizardLM-7B-Uncensored"
# 按模型自动下载/缓存分词器
tokenizer = AutoTokenizer.from_pretrained(model_name)
# 按 quant_config 做 4bit 量化加载；dtype 指定浮点精度
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    dtype=torch.float16
)
# 把模型放到 CUDA（需本机有可用 GPU）
model.to("cuda")
# 若分词器没有 pad_token，用 eos_token 顶上，避免 batch/pad 报错
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# 同步模型配置里的 pad_token_id，生成时对齐 padding 语义
model.config.pad_token_id = tokenizer.pad_token_id
# 记录推理设备字符串：有 CUDA 用 cuda，否则 cpu（后续 .to(device) 用）
device = "cuda" if torch.cuda.is_available() else "cpu"


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 86.00 MiB. GPU 0 has a total capacity of 11.59 GiB of which 49.19 MiB is free. Including non-PyTorch memory, this process has 10.98 GiB memory in use. Of the allocated memory 10.80 GiB is allocated by PyTorch, and 12.72 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# ========== 生成函数：prompt → tokenize → generate → decode ==========
def generate_synthetic_data(prompt, max_tokens=256, temperature=0.7):
    # 把文本 prompt 编成张量，并搬到前面选定的 device（GPU/CPU）
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 调用因果语言模型做续写生成
    outputs = model.generate(
        **inputs,
        # 最多新生成多少个 token（不含输入长度）
        max_new_tokens=max_tokens,
        # 温度：越高越随机，越低越保守
        temperature=temperature,
        # nucleus sampling：只在累计概率 top_p 的词里采样
        top_p=0.9
    )

    # 把输出 token id 解码回可读文本；跳过特殊符号
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
# ========== 业务 prompt：求职候选人匹配的合成 JSON 记录 ==========
# 三引号字符串会原样发给模型——字段说明与英文指令禁止翻译（影响输出结构）
prompt = """
Generate 5 synthetic job-candidate matching records in JSON format.
Include:
- candidate_id (UUID)
- candidate_name
- current_role
- years_of_experience
- technical_skills (List of strings: languages, frameworks, tools)
- resume_summary (A short paragraph describing professional background)
- applied_job_title
- job_description_requirements (Key requirements from the JD to match against)
- matching_score (A decimal between 0 and 1 representing initial fit)
- missing_skills (List of skills the candidate lacks for this specific JD)
"""


In [ ]:
# ========== 调用生成并打印结果 ==========
# 用上一格的 prompt 跑一遍 generate_synthetic_data
synthetic_data = generate_synthetic_data(prompt)
# 在笔记本输出区查看模型返回的（期望为 JSON 风格的）文本
print(synthetic_data)


In [ ]:
# ========== Gradio 简易界面：文本进、文本出 ==========
# 安装 Gradio（若环境已有可重复执行，-q 安静模式）
!pip install gradio -q
# 再次导入，确保本格可独立运行
import gradio as gr

# UI 回调：把输入框文本交给同一个生成函数
def generate_ui(prompt):
    return generate_synthetic_data(prompt)

# Interface：最简「单输入单输出」页面
interface = gr.Interface(
    fn=generate_ui,               # 要点击/提交时执行的函数
    inputs="text",                # 输入组件类型：纯文本
    outputs="text",               # 输出组件类型：纯文本
    title="Synthetic Medical Data Generator",  # 页面标题（英文保持原样）
    description="Enter a prompt to generate synthetic medical dataset in JSON format."  # 说明文案保持原样
)

# 启动用户界面（原笔记本此处未调用 launch；逻辑保持原样，不擅自补上）
